In [ ]:
#| hide
from estravon_bench.compare import compare

# estravon-backend-benchmarks

> Compare MinerU / Mistral OCR / Datalab / Replicate on YOUR PDF,
> apples-to-apples, behind one call. Drives a running `estravon-backend`
> over HTTP -- it does not extract anything itself.

**Prerequisite:** you need `estravon-backend` installed and reachable with the
engines you want to compare configured -- MinerU installed locally
(`pip install "estravon-backend[mineru]"`), and/or `MISTRAL_API_KEY` /
`DATALAB_API_KEY` / `REPLICATE_API_TOKEN` set for the cloud engines. This
package orchestrates `estravon-backend`; it never extracts anything on its
own. See `estravon-backend`'s `docs/API.md` for the engine-decision table
and how to get each engine's key.

## Install

In [ ]:
#| eval: false
!pip install estravon-backend-benchmarks

## Quickstart

`compare()` has two modes (see `estravon-backend`'s `docs/API.md` section
"Engine selection" for why there's no single-URL multi-engine mode -- engine
choice is fixed per running instance, on purpose):

- **Mode A (below, the common case):** pass `engines=[...]` and this package
  spawns one pinned `estravon --backend <engine> --port <N>` subprocess per
  engine for you -- nothing to configure beyond having `estravon-backend`
  installed with the relevant keys/local models available.
- **Mode B:** pass `engine_urls={"mistral": "http://host:port", ...}` instead,
  if you already have separately-running single-engine instances.

The cell below is marked non-executing in this rendered copy of the notebook
(no backend/engines are available in the environment that builds these
docs) -- it is exactly what you'd run locally.

In [ ]:
#| eval: false
result = compare(
    pdf_path="sample.pdf",
    page_range="1-4",
    engines=["mineru", "mistral"],   # whichever engines you have configured
    mode="balanced",
)
print(result.to_markdown_table())

This prints a side-by-side table:

```
| engine | time (s) | cost (usd) | local | pages | status |
|---|---|---|---|---|---|
| mineru | 34.10 | free (local) | yes | 4 | ok |
| mistral | 2.30 | $0.0080 | no | 4 | ok |
```

Then look at each engine's actual output:

```python
for r in result:
    print(f"--- {r.engine} ---")
    print(r.markdown[:500] if r.ok else f"ERROR: {r.error}")

print(result.diff("mineru", "mistral"))   # optional -- line diff for eyeballing
```

**⚠️ The two-step fetch is handled for you** -- `Client.fetch_markdown()`
already does the `md_url` → actual text round trip described in
`estravon-backend`'s `docs/API.md`. If you're extending `client.py`
yourself, that's the detail to know about; `compare()`'s own callers never
see a bare URL.

**Honest-cost labelling:** `local=True` engines (MinerU today) show
`cost_usd=0.0` and the table renders "free (local)" -- that means *zero
dollars*, not *zero effort or best value*. A free engine that's ten times
slower is not automatically the right choice.

**Scope:** this is an evaluation aid for comparing engines on your own PDFs
apples-to-apples -- not a production layer, and not a leaderboard. It
compares *your* PDF on *your* configured engines; it makes no claim about
which engine is best in general.

## Scoring (optional, not required to be useful)

`compare()` + `to_markdown_table()` -- outputs, time, and cost -- is the whole
value proposition and needs zero ground truth. Scoring against a reference
(TEDS for tables, CER/edit-distance for text, formula metrics) is optional
upside in `estravon_bench.score` for contributors who want it; it is not
required to get value from this package.